# ¿Hablan los programas electorales de lo que preocupa a los españoles?

Comparación de los programas de las elecciones generales de noviembre de 2019 (Ciudadanos, Podemos, PSOE, Más País, PP y Vox):

1. Nube de palabras de cada programa.
2. Qué porcentaje de cada programa dedica cada partido a los seis principales problemas según el barómetro del CIS de noviembre de 2019.

**Datos necesarios** (ver [`data/README.md`](../data/README.md)):

- `staging/programas_electorales/201910/<partido>.txt`: texto de cada programa, extraído de los PDF oficiales.
- Los problemas del CIS ya están en [`resultados/cis_nov_19.csv`](../resultados/cis_nov_19.csv).

Las nubes de palabras generadas están en [`docs/img/nubes_palabras/`](../docs/img/nubes_palabras/).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import plotly.express as px
from wordcloud import STOPWORDS, WordCloud

sys.path.insert(0, str(Path.cwd().parent))  # permite importar el paquete politicai desde notebooks/
from politicai.config import RESULTADOS_DIR, STAGING_DIR, publicar_en_chart_studio
from politicai.texto import CleanText, cobertura_temas, frecuencias

for recurso in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(recurso, quiet=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
PROGRAMAS_DIR = STAGING_DIR / "programas_electorales" / "201910"

# nombre en los gráficos: (fichero, color de fondo de la nube, paleta de las palabras)
PARTIDOS = {
    "Ciudadanos": ("ciudadanos", "darkorange", "Pastel1"),
    "Podemos": ("podemos", "mediumpurple", "Pastel1"),
    "PSOE": ("psoe", "lightcoral", "Pastel1"),
    "Más País": ("mas_pais", "aquamarine", "tab20b"),
    "PP": ("pp", "cornflowerblue", "Pastel1"),
    "Vox": ("vox", "limegreen", "Pastel1"),
}

COLORES = {"PSOE": "#E52B50", "PP": "#1E90FF", "Ciudadanos": "#FFA500",
           "Podemos": "#9370DB", "Vox": "#00FF00", "Más País": "#20FFA1"}

## Texto de los programas

In [ ]:
ct = CleanText()
programas = {}
for partido, (fichero, _, _) in PARTIDOS.items():
    texto = (PROGRAMAS_DIR / f"{fichero}.txt").read_text()
    programas[partido] = ct.do_all(texto)

## Nubes de palabras

In [ ]:
for partido, (_, fondo, paleta) in PARTIDOS.items():
    wordcloud = WordCloud(
        width=800, height=400, random_state=1, background_color=fondo, colormap=paleta,
        max_words=75, ranks_only=True, collocations=True, stopwords=STOPWORDS,
        normalize_plurals=True, min_word_length=3,
    ).generate_from_frequencies(frecuencias(programas[partido]))

    plt.figure(figsize=(20, 10))
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(partido)
    plt.show()

## Principales problemas según el CIS (noviembre de 2019)

In [ ]:
df_cis = pd.read_csv(RESULTADOS_DIR / "cis_nov_19.csv").sort_values("Porcentaje", ascending=False)
df_cis.head(6)

Cada problema se representa con una lista de palabras relacionadas. Como se comparan raíces (`do_all_stem`), "trabajo", "trabajar" y "trabajador" cuentan como la misma palabra.

In [ ]:
PALABRAS_TEMA = {
    "paro": "paro prestación erte ere desempleo inem trabajo trabajar trabajador pyme autonomo empleo desocupacion huelga salario despido destitución inactividad subsidio",
    "políticos": "político política partido acuerdo diputado aforado crispación",
    "economía": "dinero economía mercado salario empleo trabajo pensiones nómina desigualdad",
    "cataluña": "cataluña procés nacionalismo independencia independentista ruptura constitución president referéndum",
    "corrupción": "corrupción nepotismo giratorias enchufe robar juicio trama cárcel fraude sobresueldo",
    "sanidad": "sanidad salud intervención vacuna cirugía hospital medico enfermero ambulatorio operación",
}
raices_tema = {tema: set(ct.do_all_stem(palabras).split(" ")) for tema, palabras in PALABRAS_TEMA.items()}

**Cobertura**: porcentaje de palabras de cada programa que pertenecen a un tema.

> La versión original de este análisis comprobaba si cada palabra aparecía *dentro* de la cadena de texto del tema (una comparación de subcadenas), así que palabras cortas contaban aunque no estuvieran en la lista. Aquí se compara palabra a palabra, por lo que los porcentajes pueden diferir de los publicados en el artículo:
>
> | | Ciudadanos | Podemos | PSOE | Más País | PP | Vox |
> |---|---|---|---|---|---|---|
> | paro | 2.18 | 2.79 | 2.87 | 2.10 | 2.48 | 2.38 |
> | políticos | 1.11 | 1.08 | 1.59 | 1.35 | 1.12 | 1.68 |
> | economía | 2.40 | 3.26 | 2.84 | 2.23 | 2.08 | 2.00 |
> | cataluña | 1.70 | 1.20 | 1.38 | 1.38 | 1.76 | 2.33 |
> | corrupción | 0.54 | 0.50 | 0.59 | 0.47 | 0.28 | 0.54 |
> | sanidad | 1.42 | 1.24 | 1.44 | 1.62 | 1.30 | 1.24 |

In [ ]:
df_cob = pd.DataFrame({
    partido: cobertura_temas(ct.do_all_stem(texto), raices_tema)
    for partido, texto in programas.items()
})
df_cob

In [ ]:
fig_temas = px.line(
    df_cob, color_discrete_map=COLORES, template="plotly_white",
    labels={"value": "Porcentaje", "index": "Tema"},
    title="Cobertura de los problemas de los españoles (nov 19) por tema y partido político",
)
fig_temas.show()

In [ ]:
fig_partidos = px.bar(
    df_cob.T, template="plotly_white",
    labels={"value": "Porcentaje", "index": "Partido"},
    title="Cobertura de los problemas de los españoles (nov 19) por partido político y tema",
).update_xaxes(categoryorder="total descending")
fig_partidos.show()
# publicar_en_chart_studio(fig_partidos, "cobertura_problemas_partidos_nov19")